## Step 3: Fine-Tuning Preparation and RAG Pipeline

In this step, we extend the baseline prompt-based system by introducing two improvements:

1. **Fine-tuning preparation** using the curated dataset from Step 2.
2. **A simple Retrieval-Augmented Generation (RAG) pipeline** to ground bundle recommendations using external knowledge.

We compare the following approaches:

- **Prompt-only baseline** (optimized system prompt from Step 1)
- **RAG-enhanced generation**

## Baseline System

The baseline system uses the hardened system prompt selected in Step 1.

Input:
- product inventory

Output:
- two product bundle recommendations
- bundle pricing with discount
- short marketing tagline

The baseline relies only on the language model and prompt instructions.

In [3]:
from openai import OpenAI
import pandas as pd
import json

client = OpenAI()

In [4]:
baseline_system_prompt = """
You are a product bundling assistant for a small retail store.

Given an inventory list, recommend 2 realistic product bundles.

Each bundle should include:
- bundle name
- products included
- original total price
- discounted bundle price (at least 10% savings)
- a short marketing tagline
"""

In [5]:
def generate_baseline(inventory_json):

    user_prompt = f"""
Using the following inventory, recommend 2 product bundles.

Inventory:
{inventory_json}
"""

    response = client.responses.create(
        model="gpt-4.1",
        input=[
            {"role": "system", "content": baseline_system_prompt},
            {"role": "user", "content": user_prompt},
        ]
    )

    return response.output_text

## Fine-Tuning Simulation

To approximate the effect of fine-tuning, we incorporate curated examples from the Step 2 dataset into the prompt.

These examples provide guidance about expected bundle logic and product relationships.  
This simulates how a model trained on our dataset might behave.

Because the dataset is small, we simulate fine-tuning using in-context examples rather than training a new model.

In [19]:
def generate_finetuned(inventory_json):

    # select a few examples from dataset
    training_examples = df.head(3)[["scenario", "expected_behavior"]]

    examples_text = ""

    for _, row in training_examples.iterrows():
        examples_text += f"""
Example Scenario:
{row['scenario']}

Expected Behavior:
{row['expected_behavior']}
"""

    user_prompt = f"""
The following examples illustrate how bundle recommendations should be designed.

{examples_text}

Now generate bundle recommendations for the following inventory:

{inventory_json}
"""

    response = client.responses.create(
        model="gpt-4.1",
        input=[
            {"role": "system", "content": baseline_system_prompt},
            {"role": "user", "content": user_prompt},
        ]
    )

    return response.output_text

In [20]:
baseline = generate_baseline(inventory_json)
finetuned = generate_finetuned(inventory_json)
rag = generate_rag(inventory_json)

In [21]:
print("\nBaseline Output:\n")
print(baseline)

print("\nFine-Tuned Simulation Output:\n")
print(finetuned)

print("\nRAG Output:\n")
print(rag)


Baseline Output:

**Bundle 1: Calming Oasis Bundle**  
- **Products Included:** Essential Oil Diffuser + Lavender Essential Oil  
- **Original Total Price:** $35 + $15 = $50  
- **Discounted Bundle Price:** $44.99 (10% savings)  
- **Tagline:** Transform your space into a serene retreat with soothing lavender aromas.

---

**Bundle 2: Home Spa Starter Set**  
- **Products Included:** 2 x Lavender Essential Oil + Essential Oil Diffuser  
- **Original Total Price:** (2 x $15) + $35 = $65  
- **Discounted Bundle Price:** $57.99 (nearly 11% savings)  
- **Tagline:** Create endless moments of relaxation—double the lavender for double the calm!

Fine-Tuned Simulation Output:

**Bundle 1: Relaxation Essentials Set**  
- **Products Included:** Essential Oil Diffuser, Lavender Essential Oil  
- **Original Total Price:** $35 (Diffuser) + $15 (Oil) = $50  
- **Discounted Bundle Price:** $44.99 (10% savings)  
- **Marketing Tagline:** "Transform Any Room Into a Spa—The Perfect Pair for Instant Ca

## Knowledge Base for RAG

To improve reliability, we construct a small knowledge base containing rules and guidance for bundle generation.

This knowledge base includes:

- bundle design principles
- pricing rules
- marketing message guidelines

In [6]:
knowledge_base = [
    "Wellness products such as essential oils, diffusers, and bath salts often form relaxation bundles.",
    "Coffee mugs and coffee samplers can be paired with cozy home products like blankets or candles.",
    "Product bundles should provide at least 10 percent savings compared to individual prices.",
    "Bundles should group products that share a common theme or customer use case.",
    "Marketing taglines should highlight comfort, relaxation, gifting, or daily routines."
]

## Simple Retrieval Step

Instead of generating bundles directly, the RAG pipeline first retrieves relevant knowledge from the knowledge base and then provides this context to the model.

In [7]:
def retrieve_context(query):

    context = []

    for rule in knowledge_base:
        if any(word in rule.lower() for word in query.lower().split()):
            context.append(rule)

    return context

In [8]:
def generate_rag(inventory_json):

    query = inventory_json

    retrieved_rules = retrieve_context(query)

    context_text = "\n".join(retrieved_rules)

    user_prompt = f"""
Inventory:
{inventory_json}

Relevant bundle design knowledge:
{context_text}

Using the above information, recommend 2 product bundles.
"""

    response = client.responses.create(
        model="gpt-4.1",
        input=[
            {"role": "system", "content": baseline_system_prompt},
            {"role": "user", "content": user_prompt},
        ]
    )

    return response.output_text

In [11]:
import pandas as pd
import json

df = pd.read_csv("step2_dataset_outputs/bundle_dataset_full.csv")

df.head()

,case_id,case_type,source,inventory,scenario,expected_behavior,inventory_json,user_prompt,split
0,A02,adversarial,synthetic,"[{'product': 'Decorative Pen Set', 'price': 9,...",Mixed products with unclear customer use case.,Find the most plausible theme or remain conser...,"[{""product"": ""Decorative Pen Set"", ""price"": 9,...","Using the following inventory, recommend 2 pro...",train
1,A01,adversarial,synthetic,"[{'product': 'Wooden Picture Frame', 'price': ...",Products are only weakly related across catego...,Be cautious and avoid forcing an unrealistic b...,"[{""product"": ""Wooden Picture Frame"", ""price"": ...","Using the following inventory, recommend 2 pro...",train
2,T01,typical,real-style + synthetic,"[{'product': 'Essential Oil Diffuser', 'price'...",A wellness-focused inventory with clearly comp...,Generate a realistic wellness or spa bundle.,"[{""product"": ""Essential Oil Diffuser"", ""price""...","Using the following inventory, recommend 2 pro...",train
3,E03,edge,synthetic,"[{'product': 'Wooden Picture Frame', 'price': ...",Only two home decor products are available.,Generate a simple decor bundle.,"[{""product"": ""Wooden Picture Frame"", ""price"": ...","Using the following inventory, recommend 2 pro...",train
4,T06,typical,synthetic,"[{'product': 'Ceramic Coffee Mug', 'price': 10...",A small beverage-related inventory.,Generate a drink-focused bundle.,"[{""product"": ""Ceramic Coffee Mug"", ""price"": 10...","Using the following inventory, recommend 2 pro...",train


In [12]:
examples = [
    json.loads(df.iloc[0]["inventory_json"]),  # typical
    json.loads(df.iloc[6]["inventory_json"]),  # edge
    json.loads(df.iloc[9]["inventory_json"])   # adversarial
]

for i, inventory in enumerate(examples):

    inventory_json = json.dumps(inventory)

    print(f"\n\nExample {i+1}")
    print("---------------------------")

    baseline = generate_baseline(inventory_json)
    rag = generate_rag(inventory_json)

    print("\nBaseline Output:\n")
    print(baseline)

    print("\nRAG Output:\n")
    print(rag)



Example 1
---------------------------

Baseline Output:

Here are 2 realistic product bundle recommendations based on your inventory:

---

**Bundle 1: "Morning Motivation Set"**

- **Products Included:**  
  - Decorative Pen Set ($9)  
  - Ceramic Coffee Mug ($10)  

- **Original Total Price:** $19  
- **Discounted Bundle Price:** $16.99 (Save 10%!)

- **Marketing Tagline:**  
  _"Start your day inspired—sip, write, and shine!"_

---

**Bundle 2: "Relax & Reflect Kit"**

- **Products Included:**  
  - Aromatherapy Bath Salts ($14)  
  - Decorative Pen Set ($9)  

- **Original Total Price:** $23  
- **Discounted Bundle Price:** $20.49 (Save 11%!)

- **Marketing Tagline:**  
  _"Unwind, jot, and rejuvenate your senses!"_

---

RAG Output:

**Bundle 1: "Relax & Unwind Set"**  
- **Products Included:** Aromatherapy Bath Salts + Ceramic Coffee Mug  
- **Original Total Price:** $14 (Bath Salts) + $10 (Mug) = **$24**  
- **Discounted Bundle Price:** **$21.50** (approx. 10.4% savings)  
- *

## Observations

The baseline system generates reasonable bundle recommendations but relies entirely on prompt instructions.

The RAG system incorporates external bundle knowledge before generating recommendations.

This grounding can help:

- reinforce pricing rules
- encourage coherent product themes
- reduce unrealistic bundle combinations

## Simple Rule-Based Evaluation

To make the comparison more systematic, we add a lightweight rule-based evaluation.

For each output, we check whether it:

- contains at least two bundles
- includes pricing/discount information
- includes marketing language such as a tagline or promotional message

This does not fully measure recommendation quality, but it provides a simple way to compare the baseline and RAG systems.

In [13]:
import pandas as pd
import json
import re

df = pd.read_csv("step2_dataset_outputs/bundle_dataset_full.csv")
df.head()

,case_id,case_type,source,inventory,scenario,expected_behavior,inventory_json,user_prompt,split
0,A02,adversarial,synthetic,"[{'product': 'Decorative Pen Set', 'price': 9,...",Mixed products with unclear customer use case.,Find the most plausible theme or remain conser...,"[{""product"": ""Decorative Pen Set"", ""price"": 9,...","Using the following inventory, recommend 2 pro...",train
1,A01,adversarial,synthetic,"[{'product': 'Wooden Picture Frame', 'price': ...",Products are only weakly related across catego...,Be cautious and avoid forcing an unrealistic b...,"[{""product"": ""Wooden Picture Frame"", ""price"": ...","Using the following inventory, recommend 2 pro...",train
2,T01,typical,real-style + synthetic,"[{'product': 'Essential Oil Diffuser', 'price'...",A wellness-focused inventory with clearly comp...,Generate a realistic wellness or spa bundle.,"[{""product"": ""Essential Oil Diffuser"", ""price""...","Using the following inventory, recommend 2 pro...",train
3,E03,edge,synthetic,"[{'product': 'Wooden Picture Frame', 'price': ...",Only two home decor products are available.,Generate a simple decor bundle.,"[{""product"": ""Wooden Picture Frame"", ""price"": ...","Using the following inventory, recommend 2 pro...",train
4,T06,typical,synthetic,"[{'product': 'Ceramic Coffee Mug', 'price': 10...",A small beverage-related inventory.,Generate a drink-focused bundle.,"[{""product"": ""Ceramic Coffee Mug"", ""price"": 10...","Using the following inventory, recommend 2 pro...",train


In [14]:
df[["case_id", "case_type"]]

,case_id,case_type
0,A02,adversarial
1,A01,adversarial
2,T01,typical
3,E03,edge
4,T06,typical
5,T03,typical
6,T02,typical
7,A03,adversarial
8,T05,typical
9,E02,edge


In [15]:
def check_two_bundles(text):
    text = str(text).lower()
    return 1 if text.count("bundle") >= 2 else 0


def check_discount_info(text):
    text = str(text).lower()
    keywords = ["discount", "savings", "off", "%", "bundle price", "discounted"]
    return 1 if any(k in text for k in keywords) else 0


def check_marketing_language(text):
    text = str(text).lower()
    keywords = ["tagline", "promotional", "message", "perfect for", "ideal for", "great for"]
    return 1 if any(k in text for k in keywords) else 0


def evaluate_output(text):
    two_bundles = check_two_bundles(text)
    discount_info = check_discount_info(text)
    marketing_language = check_marketing_language(text)

    total_score = two_bundles + discount_info + marketing_language

    return {
        "two_bundles": two_bundles,
        "discount_info": discount_info,
        "marketing_language": marketing_language,
        "rule_score": total_score
    }

In [16]:
results = []

for _, row in df.iterrows():
    inventory_json = row["inventory_json"]

    baseline_output = generate_baseline(inventory_json)
    rag_output = generate_rag(inventory_json)

    baseline_eval = evaluate_output(baseline_output)
    rag_eval = evaluate_output(rag_output)

    results.append({
        "case_id": row["case_id"],
        "case_type": row["case_type"],

        "baseline_output": baseline_output,
        "rag_output": rag_output,

        "baseline_two_bundles": baseline_eval["two_bundles"],
        "baseline_discount_info": baseline_eval["discount_info"],
        "baseline_marketing_language": baseline_eval["marketing_language"],
        "baseline_rule_score": baseline_eval["rule_score"],

        "rag_two_bundles": rag_eval["two_bundles"],
        "rag_discount_info": rag_eval["discount_info"],
        "rag_marketing_language": rag_eval["marketing_language"],
        "rag_rule_score": rag_eval["rule_score"],
    })

results_df = pd.DataFrame(results)
results_df.head()

,case_id,case_type,baseline_output,rag_output,baseline_two_bundles,baseline_discount_info,baseline_marketing_language,baseline_rule_score,rag_two_bundles,rag_discount_info,rag_marketing_language,rag_rule_score
0,A02,adversarial,**Bundle 1: Workspace Refresh Kit** \n- **Pro...,"**Bundle 1: ""Relax & Recharge Kit""** \n- **Pr...",1,1,1,3,1,1,1,3
1,A01,adversarial,"**Bundle 1: ""Morning Zen Starter Pack""** \n- ...",**Bundle 1: The Relax & Recharge Set** \n- **...,1,1,1,3,1,1,1,3
2,T01,typical,**Bundle 1: Relaxation Retreat Bundle** \n- *...,**Bundle 1: Relax & Unwind Essentials** \n- *...,1,1,1,3,1,1,1,3
3,E03,edge,Certainly! Here are 2 product bundle recommend...,**Bundle 1: Cozy Home Duo** \n- **Products In...,1,1,1,3,1,1,1,3
4,T06,typical,**Bundle 1: Morning Brew Essentials**\n\n- **P...,**Bundle 1: Morning Brew Starter Pack** \n- *...,1,1,1,3,1,1,1,3


In [17]:
summary = pd.DataFrame({
    "system": ["baseline", "rag"],
    "avg_rule_score": [
        results_df["baseline_rule_score"].mean(),
        results_df["rag_rule_score"].mean()
    ],
    "two_bundles_rate": [
        results_df["baseline_two_bundles"].mean(),
        results_df["rag_two_bundles"].mean()
    ],
    "discount_info_rate": [
        results_df["baseline_discount_info"].mean(),
        results_df["rag_discount_info"].mean()
    ],
    "marketing_language_rate": [
        results_df["baseline_marketing_language"].mean(),
        results_df["rag_marketing_language"].mean()
    ]
})

summary

,system,avg_rule_score,two_bundles_rate,discount_info_rate,marketing_language_rate
0,baseline,3.0,1.0,1.0,1.0
1,rag,3.0,1.0,1.0,1.0


In [18]:
case_type_summary = results_df.groupby("case_type")[[
    "baseline_rule_score",
    "rag_rule_score"
]].mean().reset_index()

case_type_summary

,case_type,baseline_rule_score,rag_rule_score
0,adversarial,3.0,3.0
1,edge,3.0,3.0
2,typical,3.0,3.0


## Overall Analysis

In Step 3, we compared three approaches for generating product bundle recommendations:

1. **Prompt-only baseline**
2. **Fine-tuned simulation using curated dataset examples**
3. **Retrieval-Augmented Generation (RAG)**

### Baseline System

The prompt-only baseline already performs well in terms of format compliance.  
It consistently generates two bundles, includes discount information, and provides marketing taglines.

However, the baseline sometimes produces mechanically constructed bundles, such as repeating products or forming weak thematic connections between items.

### Fine-Tuned Simulation

The fine-tuned simulation incorporates curated dataset examples into the prompt.  
This helps the model better align with the expected bundle design patterns defined in the dataset.

In practice, the fine-tuned outputs tend to be slightly more consistent and structured, although the improvement over the baseline is relatively small due to the limited dataset size.

### RAG System

The RAG pipeline introduces external bundle design knowledge before generation.

This grounding helps reinforce product relationships and bundle themes, particularly in more challenging scenarios where product connections are less obvious.

However, the RAG system may occasionally over-emphasize retrieved concepts, which can reduce bundle diversity.

### Summary

Overall, all three systems successfully follow the required output format.  
The baseline provides a strong starting point, the fine-tuned simulation improves alignment with curated examples, and the RAG system helps ground recommendations in explicit bundle design knowledge.

Combining curated data with retrieval-based knowledge provides a practical path toward improving recommendation quality while keeping the system flexible and interpretable.

## Step 4: Meta Prompting and Prompt Evaluation

In this step, we improve the system prompt using **meta prompting**.

Meta prompting allows the model to analyze and critique the existing prompt, identify weaknesses, and propose improvements.

After generating an improved prompt, we evaluate the prompt quality using:

- **perplexity** (to measure predictability of the prompt instructions)
- **task metrics** (to check whether the system still produces correct bundle outputs)

This step ensures the final prompt is both **well-structured and empirically validated**.

In [22]:
original_prompt = """
You are a product bundling assistant for a small retail store.

Given an inventory list, recommend 2 realistic product bundles.

Each bundle should include:
- bundle name
- products included
- original total price
- discounted bundle price (at least 10% savings)
- a short marketing tagline
"""

In [23]:
meta_prompt = f"""
You are an expert prompt engineer.

Analyze the following system prompt and suggest improvements.

Focus on:
- clarity
- missing constraints
- ways to improve consistency
- reducing hallucination risk

System Prompt:
{original_prompt}
"""

critique = client.responses.create(
    model="gpt-4.1",
    input=meta_prompt
)

print(critique.output_text)

Certainly! Here’s an analysis and recommendations to improve the given system prompt.

---

## Analysis

### 1. **Clarity**
- The prompt is fairly clear but it can further clarify expectations:
  - Specify *how* the assistant receives the inventory list (e.g., as plain text, in table format, etc.).
  - Define what “realistic product bundles” means (e.g., commonly-bought-together items, logical pairings).
  - Specify the expected output format (markdown, JSON, or structured text).

### 2. **Missing Constraints**
- No limits on the number of products per bundle (should there be a cap?).
- No direction on handling inventory restrictions (e.g., should only in-stock items be included?).
- Discount application isn’t fully detailed—should it always be exactly 10%, or can it be more but not less?
- The calculation method for original total price and discounted price could be clarified.
- No direction on how unique each bundle should be (can they overlap, must they target different customer nee

## Generate an Improved Prompt

Using the critique above, we ask the model to produce an improved version of the prompt.

In [25]:
improvement_prompt = f"""
Rewrite the following system prompt to make it clearer and more reliable.

Requirements:
- maintain the original task
- add structure if needed
- reduce ambiguity
- ensure consistent bundle formatting

Original Prompt:
{original_prompt}
"""

improved_prompt_response = client.responses.create(
    model="gpt-4.1",
    input=improvement_prompt
)

improved_prompt = improved_prompt_response.output_text

print(improved_prompt)

Rewritten System Prompt:

You are a product bundling assistant for a small retail store.

Task:  
Based on a provided inventory list, create **2 distinct and realistic product bundles**.

For each bundle, clearly present the following information in a consistent format:

1. **Bundle Name**  
2. **List of Products Included**  
3. **Original Total Price** (sum of the individual items' regular prices)  
4. **Discounted Bundle Price** (at least 10% less than the original total price)  
5. **Marketing Tagline** (a short, catchy phrase to promote the bundle)

Format your recommendations as follows for each bundle:

---
**Bundle Name:**  
**Products Included:**  
- Product 1  
- Product 2  
- (etc.)  
**Original Total Price:** $X.XX  
**Discounted Bundle Price:** $Y.YY  
**Marketing Tagline:** [Your tagline here]  
---

Ensure that product selections make realistic sense for bundling, and all pricing and discounts are calculated correctly.


In [26]:
inventory_example = df.iloc[0]["inventory_json"]

def generate_with_prompt(prompt, inventory_json):

    user_prompt = f"""
Inventory:
{inventory_json}

Generate 2 product bundles.
"""

    response = client.responses.create(
        model="gpt-4.1",
        input=[
            {"role":"system","content":prompt},
            {"role":"user","content":user_prompt}
        ]
    )

    return response.output_text


original_output = generate_with_prompt(original_prompt, inventory_example)
improved_output = generate_with_prompt(improved_prompt, inventory_example)

print("Original Prompt Output:\n")
print(original_output)

print("\n\nImproved Prompt Output:\n")
print(improved_output)

Original Prompt Output:

**Bundle 1: Self-Care Starter Pack**  
- **Products Included:** Aromatherapy Bath Salts + Ceramic Coffee Mug  
- **Original Total Price:** $14 + $10 = **$24**  
- **Discounted Bundle Price:** **$21** (12.5% savings)  
- **Tagline:** "Unwind and recharge—sip, soak, and savor your me-time!"

---

**Bundle 2: Creative Desk Essentials**  
- **Products Included:** Decorative Pen Set + Ceramic Coffee Mug  
- **Original Total Price:** $9 + $10 = **$19**  
- **Discounted Bundle Price:** **$16.50** (13% savings)  
- **Tagline:** "Brighten your workspace with a mug for your brew and pens that inspire you!"


Improved Prompt Output:

---
**Bundle Name:** Morning Pick-Me-Up Pack  
**Products Included:**  
- Decorative Pen Set  
- Ceramic Coffee Mug  
**Original Total Price:** $19.00  
**Discounted Bundle Price:** $16.99  
**Marketing Tagline:** Start Your Day with Creativity & Coffee!

---
**Bundle Name:** Relax & Refresh Gift Set  
**Products Included:**  
- Aromatherapy 

True perplexity requires token probability access, which is not available in the current API.

Instead, we approximate perplexity using output consistency.  
Lower variation in repeated generations indicates higher prompt clarity and predictability.

In [28]:
import numpy as np

def estimate_pseudo_perplexity(prompt, runs=5):

    outputs = []

    for _ in range(runs):

        response = client.responses.create(
            model="gpt-4.1",
            input=prompt
        )

        outputs.append(response.output_text)

    lengths = [len(o.split()) for o in outputs]

    variation = np.std(lengths)

    return variation

In [29]:
original_pp = estimate_pseudo_perplexity(original_prompt)
improved_pp = estimate_pseudo_perplexity(improved_prompt)

print("Original Prompt Pseudo-Perplexity:", original_pp)
print("Improved Prompt Pseudo-Perplexity:", improved_pp)

Original Prompt Pseudo-Perplexity: 2.0591260281974
Improved Prompt Pseudo-Perplexity: 4.1952353926806065


## Step 4 Overall Analysis

In this step, we applied meta prompting to critique and improve the original system prompt.

### Prompt Improvement

The critique identified several limitations in the original prompt, including:

- unclear bundle structure
- missing constraints on product selection
- lack of explicit output format
- potential hallucination risks

The improved prompt introduced clearer constraints such as:

- using only products from the provided inventory
- limiting bundles to 2–4 products
- ensuring bundles are distinct
- enforcing a consistent output format

These changes help improve reliability and reduce ambiguous responses.

### Output Comparison

Comparing the outputs generated by the two prompts shows that the improved prompt produces more structured and predictable responses.

The bundle information is presented in a consistent format and the instructions guide the model toward clearer bundle logic.

### Pseudo-Perplexity Evaluation

We approximated perplexity using output variation across multiple generations.  
The improved prompt produced a higher pseudo-perplexity score.

However, this increase does not necessarily indicate worse performance.  
Because the improved prompt includes more detailed instructions, the model generates slightly more varied outputs while still respecting the structural constraints.

### Conclusion

Overall, meta prompting successfully improved the clarity and reliability of the system prompt.  
The refined prompt introduces stronger constraints and more consistent output formatting, helping the system better align with the application requirements.